# 🎵 MusicGen Generator — GPU-Powered Music Generation

Generate AI music tracks using Meta's MusicGen model on Colab's free GPU.

**How to use:**
1. Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Edit the prompts in the "Configure" section
4. Generated tracks will be saved to Google Drive

**Performance:** ~10-15 seconds per 30-second track on T4 GPU

In [ ]:
#@title 1. Install Dependencies
!pip install -q transformers torch torchaudio scipy

In [ ]:
#@title 2. Mount Google Drive (for saving tracks)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Load MusicGen Model
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import scipy.io.wavfile
import os
import time
from datetime import datetime
from IPython.display import Audio, display

# Use 'small' for speed or 'medium' for better quality
MODEL_SIZE = 'small'  #@param ['small', 'medium']

print(f'Loading MusicGen {MODEL_SIZE}...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

processor = AutoProcessor.from_pretrained(f'facebook/musicgen-{MODEL_SIZE}')
model = MusicgenForConditionalGeneration.from_pretrained(f'facebook/musicgen-{MODEL_SIZE}').to(device)
sampling_rate = model.config.audio_encoder.sampling_rate
print(f'Model loaded! Sampling rate: {sampling_rate}Hz')

In [ ]:
#@title 4. Configure Generation

# === EDIT THESE ===
GENRE = 'baby-lullaby'  #@param {type:'string'}
NUM_TRACKS = 5  #@param {type:'integer'}
DURATION_SECONDS = 30  #@param {type:'integer'}

# Prompts — one per track. Add as many as you want.
# If fewer prompts than NUM_TRACKS, they'll be reused cyclically.
PROMPTS = [
    'baby lullaby, gentle piano, soft and soothing, music box, instrumental',
    'baby lullaby, warm strings, peaceful, dreamy, instrumental',
    'baby lullaby, acoustic guitar, tender, calming, instrumental',
    'baby lullaby, harp and flute, ethereal, sleepy, instrumental',
    'baby lullaby, soft chimes, peaceful melody, slow tempo, instrumental',
]

# Output folder in Google Drive
OUTPUT_DIR = f'/content/drive/MyDrive/music-library/{GENRE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Calculate tokens needed (~50 tokens per second)
MAX_TOKENS = int(DURATION_SECONDS * 50)

print(f'Genre: {GENRE}')
print(f'Tracks: {NUM_TRACKS}')
print(f'Duration: ~{DURATION_SECONDS}s per track ({MAX_TOKENS} tokens)')
print(f'Output: {OUTPUT_DIR}')
print(f'Prompts: {len(PROMPTS)}')

In [ ]:
#@title 5. Generate Tracks! 🎶
import json

total_start = time.time()

for i in range(NUM_TRACKS):
    prompt = PROMPTS[i % len(PROMPTS)]
    track_num = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.wav') or f.endswith('.mp3')]) + 1
    filename = f'track{track_num:03d}.wav'
    filepath = os.path.join(OUTPUT_DIR, filename)
    
    print(f'\n[{i+1}/{NUM_TRACKS}] Generating: {prompt}')
    start = time.time()
    
    inputs = processor(text=[prompt], padding=True, return_tensors='pt').to(device)
    audio_values = model.generate(**inputs, max_new_tokens=MAX_TOKENS)
    
    elapsed = time.time() - start
    audio_data = audio_values[0, 0].cpu().numpy()
    duration = len(audio_data) / sampling_rate
    
    scipy.io.wavfile.write(filepath, rate=sampling_rate, data=audio_data)
    
    # Save metadata
    meta = {
        'prompt': prompt,
        'provider': f'musicgen-{MODEL_SIZE}',
        'genre': GENRE,
        'duration': round(duration, 1),
        'created_at': datetime.now().isoformat(),
    }
    meta_path = filepath.replace('.wav', '.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f'  ✓ {filename} — {duration:.1f}s, {size_mb:.1f}MB, generated in {elapsed:.1f}s')
    
    # Play the last generated track
    if i == NUM_TRACKS - 1 or i == 0:
        display(Audio(audio_data, rate=sampling_rate))

total_elapsed = time.time() - total_start
print(f'\n=== Done! {NUM_TRACKS} tracks in {total_elapsed:.0f}s ===')
print(f'Saved to: {OUTPUT_DIR}')
print(f'Avg: {total_elapsed/NUM_TRACKS:.1f}s per track')

---
## Batch Mode — Generate Multiple Genres
Run the cell below to generate tracks for multiple genres at once.

In [ ]:
#@title 6. (Optional) Batch Generate Multiple Genres

BATCH_CONFIG = {
    'lofi-hiphop': {
        'count': 5,
        'duration': 60,
        'prompts': [
            'lofi hip hop, chill beats, vinyl crackle, soft piano, mellow, instrumental',
            'lofi hip hop, jazzy chords, warm bass, tape hiss, relaxing, instrumental',
            'lofi hip hop, rainy mood, gentle keys, downtempo, cozy, instrumental',
        ]
    },
    'baby-lullaby': {
        'count': 5,
        'duration': 60,
        'prompts': [
            'baby lullaby, gentle piano, music box, soft and soothing, instrumental',
            'baby lullaby, warm strings, harp, peaceful, dreamy, instrumental',
            'baby lullaby, acoustic guitar, tender, calming melody, instrumental',
        ]
    },
    'dark-ambient': {
        'count': 5,
        'duration': 60,
        'prompts': [
            'dark ambient, deep drone, mysterious, atmospheric, cinematic, instrumental',
            'dark ambient, eerie pads, haunting, slow evolving textures, instrumental',
            'dark ambient, industrial, distant thunder, low rumble, instrumental',
        ]
    },
    'relaxing-piano': {
        'count': 5,
        'duration': 60,
        'prompts': [
            'relaxing piano, classical, gentle, flowing melody, peaceful, instrumental',
            'relaxing piano, soft chords, contemplative, serene, instrumental',
            'relaxing piano, impressionist, debussy style, dreamy, instrumental',
        ]
    },
}

DRIVE_BASE = '/content/drive/MyDrive/music-library'

total_tracks = sum(cfg['count'] for cfg in BATCH_CONFIG.values())
print(f'Generating {total_tracks} tracks across {len(BATCH_CONFIG)} genres...\n')

grand_start = time.time()

for genre, cfg in BATCH_CONFIG.items():
    out_dir = os.path.join(DRIVE_BASE, genre)
    os.makedirs(out_dir, exist_ok=True)
    max_tokens = int(cfg['duration'] * 50)
    
    print(f'\n=== {genre} ({cfg["count"]} tracks, ~{cfg["duration"]}s each) ===')
    
    for i in range(cfg['count']):
        prompt = cfg['prompts'][i % len(cfg['prompts'])]
        track_num = len([f for f in os.listdir(out_dir) if f.endswith('.wav')]) + 1
        filename = f'track{track_num:03d}.wav'
        filepath = os.path.join(out_dir, filename)
        
        start = time.time()
        inputs = processor(text=[prompt], padding=True, return_tensors='pt').to(device)
        audio_values = model.generate(**inputs, max_new_tokens=max_tokens)
        elapsed = time.time() - start
        
        audio_data = audio_values[0, 0].cpu().numpy()
        duration = len(audio_data) / sampling_rate
        scipy.io.wavfile.write(filepath, rate=sampling_rate, data=audio_data)
        
        meta = {'prompt': prompt, 'provider': f'musicgen-{MODEL_SIZE}', 'genre': genre, 'duration': round(duration, 1), 'created_at': datetime.now().isoformat()}
        with open(filepath.replace('.wav', '.json'), 'w') as f:
            json.dump(meta, f, indent=2)
        
        print(f'  [{i+1}/{cfg["count"]}] {filename} — {duration:.1f}s in {elapsed:.1f}s')

grand_elapsed = time.time() - grand_start
print(f'\n=== All done! {total_tracks} tracks in {grand_elapsed:.0f}s ===')
print(f'Saved to: {DRIVE_BASE}/')

In [ ]:
#@title 7. (Optional) Download tracks as ZIP
import shutil
from google.colab import files

DOWNLOAD_GENRE = 'baby-lullaby'  #@param {type:'string'}

src = f'/content/drive/MyDrive/music-library/{DOWNLOAD_GENRE}'
zip_path = f'/content/{DOWNLOAD_GENRE}'

if os.path.exists(src):
    shutil.make_archive(zip_path, 'zip', src)
    files.download(f'{zip_path}.zip')
    print(f'Downloading {DOWNLOAD_GENRE}.zip')
else:
    print(f'No tracks found for {DOWNLOAD_GENRE}. Generate some first!')